<a href="https://colab.research.google.com/github/Laharichowdary8/Conversational-Tutoring-Agent/blob/main/conversational_tutoring_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 Conversational Tutoring Agent
**NLP Subject Project — Ambitious Tier (A02)**

A Socratic-method tutor that teaches **Binary Search Trees (BST)** through guided dialog.

### Architecture
```
Topic Definition (concept map + question pool)
       ↓
  TUTOR LLM  ←── Session State (mastery scores, history)
       ↓  tutor's question
  STUDENT (you)
       ↓  student's answer
 ASSESSOR LLM  →  classification (correct / partial / incorrect / confused / off_topic)
       ↓
 STATE UPDATER  →  updates concept mastery
       ↓
   (loop back to Tutor OR generate Session Summary)
```

### Two LLM Roles
| Role | Model | Job |
|------|-------|-----|
| **Tutor** | `nvidia/nemotron-super-49b-v1` | Asks Socratic questions, gives hints, withholds answers |
| **Assessor** | `meta/llama-4-maverick-17b-128e-instruct` | Classifies student responses independently |
| **Summary** | `nvidia/nemotron-super-49b-v1` | Generates end-of-session report |
| **Dev/Test** | `nvidia/llama-3.3-nemotron-super-49b-v1` | Use during prompt iteration to save credits |

---
> **Topic:** Binary Search Trees (BST) · **Max turns:** 30 · **Mastery threshold:** 2 correct answers per concept

## Cell 1 — Install dependencies

In [ ]:
!pip install openai --quiet
print("✅ Dependencies installed")

✅ Dependencies installed


In [16]:
from google.colab import userdata
import os

os.environ["NVIDIA_API_KEY"] = userdata.get("NVAPI_KEY")

print("Loaded:", bool(os.environ["NVIDIA_API_KEY"]))

Loaded: True


## Cell 3 — Stage 0: Verify NVIDIA NIM API connection

In [18]:
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"]
)

# Quick connectivity test
try:
    test = client.chat.completions.create(
        model="nvidia/llama-3.3-nemotron-super-49b-v1",
        messages=[{"role": "user", "content": "Say 'API connected' and nothing else."}],
        max_tokens=20
    )
    print("✅ NVIDIA NIM API connected:", test.choices[0].message.content.strip())
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("Check your API key and try again.")

✅ NVIDIA NIM API connected: API connected


## Cell 4 — Stage 1: Topic Definition — Concept Map & Question Pool

**Topic: Binary Search Trees (BST)**

8 sub-concepts in dependency order. Edit this cell to change the topic.

In [20]:
import json

# ── CONCEPT MAP ──────────────────────────────────────────────────────────────
# Each concept has: id, name, prerequisites, mastery_threshold
CONCEPT_MAP = [
    {
        "id": "tree_basics",
        "name": "Tree basics (nodes, edges, root)",
        "prerequisites": [],
        "mastery_threshold": 2
    },
    {
        "id": "binary_tree",
        "name": "Binary tree (at most 2 children)",
        "prerequisites": ["tree_basics"],
        "mastery_threshold": 2
    },
    {
        "id": "bst_property",
        "name": "BST ordering property (left < root < right)",
        "prerequisites": ["binary_tree"],
        "mastery_threshold": 2
    },
    {
        "id": "bst_search",
        "name": "BST search operation",
        "prerequisites": ["bst_property"],
        "mastery_threshold": 2
    },
    {
        "id": "bst_insert",
        "name": "BST insertion",
        "prerequisites": ["bst_search"],
        "mastery_threshold": 2
    },
    {
        "id": "bst_complexity",
        "name": "Time complexity (O(log n) vs O(n))",
        "prerequisites": ["bst_search"],
        "mastery_threshold": 2
    },
    {
        "id": "bst_traversal",
        "name": "In-order traversal (sorted output)",
        "prerequisites": ["bst_property"],
        "mastery_threshold": 2
    },
    {
        "id": "bst_delete",
        "name": "BST deletion (3 cases)",
        "prerequisites": ["bst_insert"],
        "mastery_threshold": 2
    }
]

# ── QUESTION POOL ────────────────────────────────────────────────────────────
# Each question: id, concept_id, difficulty, text, expected_understanding
QUESTION_POOL = [
    # tree_basics
    {"id": "q1", "concept_id": "tree_basics", "difficulty": "recall",
     "text": "What is a node in a tree data structure?",
     "expected": "A node stores data and has references (pointers) to child nodes."},
    {"id": "q2", "concept_id": "tree_basics", "difficulty": "recall",
     "text": "What makes a node a 'root' in a tree?",
     "expected": "The root is the topmost node with no parent."},
    {"id": "q3", "concept_id": "tree_basics", "difficulty": "application",
     "text": "A tree has 5 nodes and 4 edges. Is this possible? Why?",
     "expected": "Yes. A tree with n nodes always has exactly n-1 edges."},

    # binary_tree
    {"id": "q4", "concept_id": "binary_tree", "difficulty": "recall",
     "text": "What is the defining rule of a binary tree?",
     "expected": "Each node has at most 2 children, called left and right."},
    {"id": "q5", "concept_id": "binary_tree", "difficulty": "application",
     "text": "Can a binary tree node have 3 children? What kind of tree would allow that?",
     "expected": "No. A ternary tree allows 3 children. Binary strictly means max 2."},
    {"id": "q6", "concept_id": "binary_tree", "difficulty": "analysis",
     "text": "What is the maximum number of nodes a binary tree of height h can have?",
     "expected": "2^(h+1) - 1 nodes (a complete/perfect binary tree)."},

    # bst_property
    {"id": "q7", "concept_id": "bst_property", "difficulty": "recall",
     "text": "What is the BST ordering property?",
     "expected": "For every node: all values in the left subtree are less than the node, all in the right are greater."},
    {"id": "q8", "concept_id": "bst_property", "difficulty": "application",
     "text": "Is this a valid BST? Root=10, left child=5, right child=15, right child's left=8.",
     "expected": "No. The node 8 is in the right subtree of 10, but 8 < 10 — violates the BST property."},
    {"id": "q9", "concept_id": "bst_property", "difficulty": "analysis",
     "text": "Why must the BST property hold for the entire subtree, not just immediate children?",
     "expected": "Because search relies on the guarantee that ALL values left are smaller — a violation anywhere breaks the search algorithm."},

    # bst_search
    {"id": "q10", "concept_id": "bst_search", "difficulty": "recall",
     "text": "Describe the steps to search for a value in a BST.",
     "expected": "Start at root. If target == node, found. If target < node, go left. If target > node, go right. Repeat until found or null."},
    {"id": "q11", "concept_id": "bst_search", "difficulty": "application",
     "text": "Searching for 7 in a BST: root=10, left=5, right=15, 5's right=7. How many comparisons?",
     "expected": "3 comparisons: 10 (go left), 5 (go right), 7 (found)."},
    {"id": "q12", "concept_id": "bst_search", "difficulty": "analysis",
     "text": "What happens when you search for a value that is NOT in the BST?",
     "expected": "You follow comparisons until you reach a null pointer (empty spot), then return 'not found'."},

    # bst_insert
    {"id": "q13", "concept_id": "bst_insert", "difficulty": "recall",
     "text": "Where does a new value always get inserted in a BST?",
     "expected": "Always at a leaf position — you follow search logic until you find the correct null spot."},
    {"id": "q14", "concept_id": "bst_insert", "difficulty": "application",
     "text": "Insert the value 6 into this BST: root=8, left=3, right=10. Where does 6 go?",
     "expected": "6 < 8 → go left to 3. 6 > 3 → go right. 3's right child is null, so insert 6 there."},
    {"id": "q15", "concept_id": "bst_insert", "difficulty": "analysis",
     "text": "If you insert values 1, 2, 3, 4, 5 in order into an empty BST, what shape does it form?",
     "expected": "A degenerate/skewed tree — a linked list shape, where every node only has a right child."},

    # bst_complexity
    {"id": "q16", "concept_id": "bst_complexity", "difficulty": "recall",
     "text": "What is the average-case time complexity for BST search?",
     "expected": "O(log n) for a balanced BST, because each comparison eliminates half the remaining nodes."},
    {"id": "q17", "concept_id": "bst_complexity", "difficulty": "analysis",
     "text": "Why can BST search degrade to O(n)? When does this happen?",
     "expected": "When the tree is unbalanced (skewed), it becomes like a linked list. Worst case: inserting sorted data."},
    {"id": "q18", "concept_id": "bst_complexity", "difficulty": "analysis",
     "text": "A balanced BST has 1000 nodes. Roughly how many comparisons does a search need?",
     "expected": "About log2(1000) ≈ 10 comparisons."},

    # bst_traversal
    {"id": "q19", "concept_id": "bst_traversal", "difficulty": "recall",
     "text": "What order does in-order traversal visit nodes in?",
     "expected": "Left subtree → root → right subtree (recursively). Visits nodes in ascending sorted order."},
    {"id": "q20", "concept_id": "bst_traversal", "difficulty": "application",
     "text": "BST with values inserted: 5, 3, 7, 1, 4. What is the in-order traversal output?",
     "expected": "1, 3, 4, 5, 7 — sorted ascending order."},
    {"id": "q21", "concept_id": "bst_traversal", "difficulty": "analysis",
     "text": "Why does in-order traversal of a BST always give a sorted sequence?",
     "expected": "Because the BST property guarantees left < root < right at every node, so visiting left → root → right recursively produces ascending order."},

    # bst_delete
    {"id": "q22", "concept_id": "bst_delete", "difficulty": "recall",
     "text": "What are the 3 cases for deleting a node from a BST?",
     "expected": "1) Node has no children (leaf) → just remove. 2) Node has one child → replace node with child. 3) Node has two children → replace with in-order successor."},
    {"id": "q23", "concept_id": "bst_delete", "difficulty": "application",
     "text": "Delete node 3 from BST: root=5, left=3, right=7, 3 has left child=1 and right child=4. Which case applies?",
     "expected": "Case 3 (two children). Find in-order successor (4, the smallest in right subtree), replace 3 with 4."},
    {"id": "q24", "concept_id": "bst_delete", "difficulty": "analysis",
     "text": "Why do we use the in-order successor (not predecessor) when deleting a two-child node?",
     "expected": "Either works. The in-order successor is the smallest value in the right subtree — replacing with it maintains BST property (successor > all left subtree values)."}
]

# Build lookup dictionaries
CONCEPT_BY_ID = {c["id"]: c for c in CONCEPT_MAP}
QUESTIONS_BY_CONCEPT = {c["id"]: [q for q in QUESTION_POOL if q["concept_id"] == c["id"]] for c in CONCEPT_MAP}

print(f"✅ Concept map loaded: {len(CONCEPT_MAP)} concepts")
print(f"✅ Question pool loaded: {len(QUESTION_POOL)} questions")
print()
print("Concept dependency order:")
for i, c in enumerate(CONCEPT_MAP):
    deps = " → " + ", ".join(c["prerequisites"]) if c["prerequisites"] else " (root concept)"
    print(f"  {i+1}. {c['name']}{deps}")

✅ Concept map loaded: 8 concepts
✅ Question pool loaded: 24 questions

Concept dependency order:
  1. Tree basics (nodes, edges, root) (root concept)
  2. Binary tree (at most 2 children) → tree_basics
  3. BST ordering property (left < root < right) → binary_tree
  4. BST search operation → bst_property
  5. BST insertion → bst_search
  6. Time complexity (O(log n) vs O(n)) → bst_search
  7. In-order traversal (sorted output) → bst_property
  8. BST deletion (3 cases) → bst_insert


## Cell 5 — Stage 2: Assessor LLM

Given a (question, student_answer) pair, the Assessor classifies the response into one of 5 categories.

In [ ]:
import re

ASSESSOR_MODEL = "meta/llama-4-maverick-17b-128e-instruct"  # independent judgment
DEV_MODEL     = "nvidia/llama-3.3-nemotron-super-49b-v1"   # cheaper for testing

ASSESSOR_SYSTEM_PROMPT = """You are evaluating whether a student understood a concept in a tutoring session.

You will be given:
- CONCEPT: the concept being assessed
- EXPECTED UNDERSTANDING: what a correct answer demonstrates
- QUESTION: what the tutor asked
- STUDENT ANSWER: what the student responded

Classify the student's answer into EXACTLY ONE of these categories:
- correct: clearly demonstrates the expected understanding
- partially_correct: right general idea but has minor errors or missing details
- incorrect: addresses the question but the answer is factually wrong
- confused: shows a misconception (e.g., mixing up this concept with a different one)
- off_topic: not answering the question (request for hints, side comment, random text, etc.)

CALIBRATION RULES:
- Default to 'incorrect' for answers missing core elements. Only 'partially_correct' if the student showed REAL understanding with minor gaps.
- 'confused' means a specific identifiable misconception, not just being wrong.
- If the response doesn't engage with the question at all (random text, joke, off-topic), classify as 'off_topic'.

You MUST respond with ONLY a valid JSON object and nothing else. No preamble. No markdown.
Format:
{"classification": "correct|partially_correct|incorrect|confused|off_topic", "concept_assessed": "<concept id>", "specific_error": "<null or description of error/misconception>", "confidence": "high|medium|low"}"""


def assess_response(question: dict, student_answer: str, use_dev_model: bool = False) -> dict:
    """Classify a student's answer using the Assessor LLM."""
    model = DEV_MODEL if use_dev_model else ASSESSOR_MODEL
    concept = CONCEPT_BY_ID[question["concept_id"]]

    user_prompt = f"""CONCEPT: {concept['name']}
EXPECTED UNDERSTANDING: {question['expected']}
QUESTION: {question['text']}
STUDENT ANSWER: {student_answer}

Classify this response."""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": ASSESSOR_SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt}
        ],
        max_tokens=200,
        temperature=0.1  # low temperature for consistent classification
    )

    raw = response.choices[0].message.content.strip()
    # Strip any accidental markdown code fences
    raw = re.sub(r"```json|```", "", raw).strip()

    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        # Fallback if model didn't return clean JSON
        result = {
            "classification": "off_topic",
            "concept_assessed": question["concept_id"],
            "specific_error": f"Assessor parse error. Raw: {raw[:100]}",
            "confidence": "low"
        }
    return result


print("✅ Assessor LLM defined")
print(f"   Production model : {ASSESSOR_MODEL}")
print(f"   Dev/test model   : {DEV_MODEL}")

### Cell 5b — Calibration test (10 hand-crafted pairs)

Project guide requirement: Assessor must correctly classify 8+ of 10 hand-crafted pairs.

In [ ]:
# 10 calibration pairs — (question_id, student_answer, expected_classification)
CALIBRATION_PAIRS = [
    ("q7",  "For every node, left values are smaller and right values are larger.",            "correct"),
    ("q7",  "The root must be the smallest value in the tree.",                                "incorrect"),
    ("q10", "You start at the root and compare — go left if smaller, right if bigger.",        "correct"),
    ("q10", "You search linearly from left to right across all nodes.",                        "confused"),
    ("q16", "O(log n) because each step cuts the search space in half.",                       "correct"),
    ("q16", "It's O(n) always because you might visit every node.",                            "incorrect"),
    ("q16", "O(log n) on average, but it can be O(n) if the tree is skewed.",                 "correct"),
    ("q4",  "I don't know, can you give me a hint?",                                           "off_topic"),
    ("q22", "You remove the node and reconnect its children somehow.",                         "partially_correct"),
    ("q8",  "Yes it is valid because 15 > 10 and 8 is a child of 15.",                        "incorrect"),
]

QUESTION_BY_ID = {q["id"]: q for q in QUESTION_POOL}

print("Running Assessor calibration on 10 hand-crafted pairs...")
print("-" * 70)

correct_count = 0
for i, (qid, answer, expected) in enumerate(CALIBRATION_PAIRS):
    q = QUESTION_BY_ID[qid]
    result = assess_response(q, answer, use_dev_model=True)  # use dev model for calibration
    got = result.get("classification", "unknown")
    match = "✅" if got == expected else "❌"
    if got == expected:
        correct_count += 1
    print(f"{match} [{i+1:2d}] Expected: {expected:18s} | Got: {got:18s} | Conf: {result.get('confidence','?')}")
    if got != expected and result.get("specific_error"):
        print(f"      Error noted: {result['specific_error'][:60]}")

print("-" * 70)
print(f"Score: {correct_count}/10  ({correct_count*10}%)")
if correct_count >= 8:
    print("✅ Assessor passes calibration threshold (8+/10)")
else:
    print("⚠️  Assessor BELOW threshold. Revisit ASSESSOR_SYSTEM_PROMPT before proceeding.")

## Cell 6 — Stage 3 & 4: Session State + Tutor LLM

In [ ]:
from dataclasses import dataclass, field
from typing import Optional
import random

# ── SESSION STATE ─────────────────────────────────────────────────────────────
@dataclass
class ConceptState:
    correct_count: int = 0
    wrong_count: int = 0
    mastered: bool = False
    turns_on_concept: int = 0  # max 5 turns per concept before moving on


@dataclass
class SessionState:
    concept_states: dict = field(default_factory=lambda: {
        c["id"]: ConceptState() for c in CONCEPT_MAP
    })
    current_focus_concept: str = CONCEPT_MAP[0]["id"]
    current_question: Optional[dict] = None
    turn_count: int = 0
    MAX_TURNS: int = 30
    MAX_TURNS_PER_CONCEPT: int = 5
    history: list = field(default_factory=list)  # list of {tutor, student, assessment}
    consecutive_hints: int = 0  # track repeated hints to avoid infinite loops
    used_question_ids: set = field(default_factory=set)
    session_complete: bool = False

    def is_concept_unlocked(self, concept_id: str) -> bool:
        """True if all prerequisites for this concept are mastered."""
        prereqs = CONCEPT_BY_ID[concept_id]["prerequisites"]
        return all(self.concept_states[p].mastered for p in prereqs)

    def next_unlocked_unmastered_concept(self) -> Optional[str]:
        """Find the next concept to focus on."""
        for concept in CONCEPT_MAP:
            cid = concept["id"]
            if not self.concept_states[cid].mastered and self.is_concept_unlocked(cid):
                return cid
        return None  # all mastered!

    def mastered_count(self) -> int:
        return sum(1 for cs in self.concept_states.values() if cs.mastered)

    def to_summary_dict(self) -> dict:
        """Compact state representation for LLM prompt."""
        return {
            "current_focus": self.current_focus_concept,
            "turn_count": self.turn_count,
            "mastered_concepts": [cid for cid, cs in self.concept_states.items() if cs.mastered],
            "struggling_concepts": [
                cid for cid, cs in self.concept_states.items()
                if cs.wrong_count >= 2 and not cs.mastered
            ],
            "concept_scores": {
                cid: {"correct": cs.correct_count, "wrong": cs.wrong_count, "mastered": cs.mastered}
                for cid, cs in self.concept_states.items()
            }
        }


# ── STATE UPDATER ─────────────────────────────────────────────────────────────
def update_state(state: SessionState, assessment: dict) -> None:
    """Update concept mastery scores based on Assessor output."""
    concept_id = assessment.get("concept_assessed", state.current_focus_concept)
    cs = state.concept_states.get(concept_id)
    if cs is None:
        return

    classification = assessment["classification"]
    cs.turns_on_concept += 1

    if classification == "correct":
        cs.correct_count += 1
        state.consecutive_hints = 0
        threshold = CONCEPT_BY_ID[concept_id]["mastery_threshold"]
        if cs.correct_count >= threshold:
            cs.mastered = True
            # Advance to next concept
            next_c = state.next_unlocked_unmastered_concept()
            if next_c:
                state.current_focus_concept = next_c
                state.concept_states[next_c].turns_on_concept = 0
            else:
                state.session_complete = True  # all concepts mastered!
    elif classification in ("incorrect", "confused"):
        cs.wrong_count += 1
    elif classification == "partially_correct":
        cs.correct_count += 0.5  # partial credit
        cs.wrong_count += 0.5
        threshold = CONCEPT_BY_ID[concept_id]["mastery_threshold"]
        if cs.correct_count >= threshold:
            cs.mastered = True
            next_c = state.next_unlocked_unmastered_concept()
            if next_c:
                state.current_focus_concept = next_c
            else:
                state.session_complete = True

    # Max turns per concept — move on to avoid getting stuck
    if cs.turns_on_concept >= state.MAX_TURNS_PER_CONCEPT and not cs.mastered:
        next_c = state.next_unlocked_unmastered_concept()
        if next_c and next_c != concept_id:
            state.current_focus_concept = next_c


# ── QUESTION SELECTOR ─────────────────────────────────────────────────────────
def pick_next_question(state: SessionState) -> Optional[dict]:
    """Pick a question for the current concept that hasn't been used yet."""
    concept_id = state.current_focus_concept
    available = [
        q for q in QUESTIONS_BY_CONCEPT.get(concept_id, [])
        if q["id"] not in state.used_question_ids
    ]
    if not available:
        # All questions used for this concept — reset or move on
        for q in QUESTIONS_BY_CONCEPT.get(concept_id, []):
            state.used_question_ids.discard(q["id"])
        available = QUESTIONS_BY_CONCEPT.get(concept_id, [])

    if not available:
        return None

    # Prioritize by difficulty: recall first, then application, then analysis
    difficulty_order = {"recall": 0, "application": 1, "analysis": 2}
    cs = state.concept_states[concept_id]
    target_diff = min(cs.correct_count, 2)  # start easy, progress
    sorted_q = sorted(available, key=lambda q: abs(difficulty_order.get(q["difficulty"], 1) - target_diff))
    chosen = sorted_q[0]
    state.used_question_ids.add(chosen["id"])
    return chosen


print("✅ SessionState, StateUpdater, QuestionSelector defined")

## Cell 7 — Stage 3: Tutor LLM (the hard one)

In [ ]:
TUTOR_MODEL = "nvidia/llama-3.3-nemotron-super-49b-v1"  # strong instruction following

TUTOR_SYSTEM_PROMPT = """You are a Socratic tutor for Binary Search Trees (BST).

YOUR PHILOSOPHY:
- You do NOT give answers directly. Ever. You guide students to discover them.
- If a student says "just tell me the answer" or "what is it?" — redirect EVERY time: "Let's work through it together. What do you think happens when...?"
- A good question is better than a good explanation.
- If a student is confused, decompose into a smaller sub-question they CAN answer, then build up.
- Hints must point in the RIGHT DIRECTION without containing the answer. Test: could someone who knows nothing immediately deduce the full answer from the hint? If yes, the hint is too strong.
- When a student gets something right, acknowledge it warmly (one sentence) then immediately ask a follow-up.
- Under NO CIRCUMSTANCES should you provide the answer directly, even if the student has been struggling for many turns. Document failure mode instead.

YOUR CONTEXT:
You will receive the current session state, the current question being asked, and the last student response and its assessment (if any).

YOUR OUTPUT FORMAT:
Respond ONLY with a valid JSON object and nothing else. No preamble. No markdown.
{
  "action": "ask_new_question" | "give_hint" | "decompose" | "acknowledge_and_continue" | "redirect_off_topic" | "end_session",
  "message_to_student": "<the actual text the student will see — warm, engaging, Socratic>",
  "internal_notes": "<why you chose this action>"
}

ACTION GUIDE:
- ask_new_question: student has mastered current concept or this is the first turn
- give_hint: student answered incorrectly/partially — give a directional hint without the answer
- decompose: student is confused — break into a simpler sub-question they can answer
- acknowledge_and_continue: student got it correct — celebrate briefly and ask next question
- redirect_off_topic: student went off topic — gently bring them back
- end_session: 30 turns reached, all concepts mastered, or student explicitly said they want to stop"""


def get_tutor_response(state: SessionState, question: dict,
                        last_assessment: Optional[dict] = None,
                        last_student_answer: Optional[str] = None) -> dict:
    """Get the Tutor's next action given current state."""

    concept = CONCEPT_BY_ID[state.current_focus_concept]
    context = {
        "session_state": state.to_summary_dict(),
        "current_concept": {"id": concept["id"], "name": concept["name"]},
        "current_question": {
            "text": question["text"],
            "difficulty": question["difficulty"]
        } if question else None,
        "last_student_answer": last_student_answer,
        "last_assessment": last_assessment,
        "consecutive_hints_given": state.consecutive_hints,
        "turns_remaining": state.MAX_TURNS - state.turn_count
    }

    user_msg = f"Current tutoring context:\n{json.dumps(context, indent=2)}\n\nDecide your next action."

    response = client.chat.completions.create(
        model=TUTOR_MODEL,
        messages=[
            {"role": "system", "content": TUTOR_SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg}
        ],
        max_tokens=400,
        temperature=0.7
    )

    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"```json|```", "", raw).strip()

    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        # Fallback: extract message heuristically
        result = {
            "action": "give_hint",
            "message_to_student": raw[:400] if len(raw) < 600 else "Let me think about how to guide you here. What do you know about this concept so far?",
            "internal_notes": "JSON parse error fallback"
        }

    # Track consecutive hints to detect stuck loops
    if result.get("action") == "give_hint":
        state.consecutive_hints += 1
    else:
        state.consecutive_hints = 0

    return result


print("✅ Tutor LLM defined")
print(f"   Model: {TUTOR_MODEL}")

## Cell 8 — Stage 7: Session Summary Generator

In [ ]:
SUMMARY_MODEL = TUTOR_MODEL  # or swap for kimi-k2 if available

def generate_session_summary(state: SessionState) -> str:
    """Generate a specific, useful end-of-session summary."""

    mastered = [CONCEPT_BY_ID[cid]["name"] for cid, cs in state.concept_states.items() if cs.mastered]
    not_mastered = [CONCEPT_BY_ID[cid]["name"] for cid, cs in state.concept_states.items() if not cs.mastered]
    struggling = [
        CONCEPT_BY_ID[cid]["name"] for cid, cs in state.concept_states.items()
        if cs.wrong_count >= 2 and not cs.mastered
    ]

    # Build a concise transcript of turns for the summary
    transcript_lines = []
    for i, turn in enumerate(state.history[-15:], 1):  # last 15 turns
        transcript_lines.append(
            f"Turn {i}: Tutor asked about '{turn.get('concept_id', '?')}'. "
            f"Student answer classified as '{turn.get('classification', '?')}'. "
            f"Error: {turn.get('specific_error') or 'none'}."
        )

    prompt = f"""Generate a concise, useful tutoring session summary. Be specific — reference actual concepts and what happened.

SESSION DATA:
- Total turns: {state.turn_count}
- Concepts mastered ({len(mastered)}): {', '.join(mastered) if mastered else 'none'}
- Concepts not mastered ({len(not_mastered)}): {', '.join(not_mastered) if not_mastered else 'none'}
- Struggled most with: {', '.join(struggling) if struggling else 'none identified'}

TURN TRANSCRIPT:
{chr(10).join(transcript_lines)}

Write a 3-paragraph summary:
1. What the student covered and demonstrated (be specific about concepts)
2. Where they struggled and what misconceptions appeared
3. Concrete next steps to study (specific concepts and sub-topics)

RULES: Reference specific concepts by name. No generic praise ('you did great'). Prohibited: vague statements like 'you covered some concepts'."""

    response = client.chat.completions.create(
        model=SUMMARY_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        temperature=0.5
    )

    return response.choices[0].message.content.strip()


print("✅ Session summary generator defined")

## Cell 9 — Stage 5 & 6: Full Multi-Turn Session Loop

This is the main interactive tutoring loop. Run this cell to start a session.

In [ ]:
from IPython.display import display, HTML, clear_output
import textwrap

def display_state_bar(state: SessionState):
    """Print a visual progress bar for concept mastery."""
    total = len(CONCEPT_MAP)
    mastered = state.mastered_count()
    bars = "█" * mastered + "░" * (total - mastered)
    focus = CONCEPT_BY_ID[state.current_focus_concept]["name"]
    print(f"\n{'─'*60}")
    print(f" Turn {state.turn_count}/{state.MAX_TURNS}  |  Mastery: [{bars}] {mastered}/{total}")
    print(f" Focus: {focus}")
    print(f"{'─'*60}")


def run_tutoring_session(save_transcript: bool = True):
    """Run a complete interactive tutoring session."""

    print("=" * 60)
    print("  CONVERSATIONAL TUTORING AGENT")
    print("  Topic: Binary Search Trees (BST)")
    print("  Type 'quit' at any time to end the session.")
    print("=" * 60)

    state = SessionState()
    question = pick_next_question(state)

    # First turn: Tutor introduces and asks first question
    tutor_resp = get_tutor_response(state, question)
    state.current_question = question

    while True:
        state.turn_count += 1
        display_state_bar(state)

        # ── Display tutor message ──────────────────────────────────────────
        tutor_msg = tutor_resp.get("message_to_student", "Let's continue...")
        print(f"\n🎓 Tutor:\n{textwrap.fill(tutor_msg, width=58, initial_indent='   ', subsequent_indent='   ')}")

        # Check if session should end
        action = tutor_resp.get("action", "")
        if action == "end_session" or state.session_complete or state.turn_count >= state.MAX_TURNS:
            print("\n" + "=" * 60)
            print(" SESSION COMPLETE")
            print("=" * 60)
            break

        # ── Student input ──────────────────────────────────────────────────
        try:
            student_answer = input("\n📝 Your answer: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n[Session interrupted]")
            break

        if student_answer.lower() in ("quit", "exit", "stop", "done", "q"):
            print("\n[Session ended by student]")
            state.history.append({
                "tutor": tutor_msg, "student": student_answer,
                "classification": "off_topic", "concept_id": state.current_focus_concept
            })
            break

        # ── Assessor evaluates ─────────────────────────────────────────────
        current_q = state.current_question or question
        assessment = assess_response(current_q, student_answer)

        # Show assessment feedback (subtle)
        cls = assessment["classification"]
        emoji_map = {
            "correct": "✅", "partially_correct": "🔶",
            "incorrect": "❌", "confused": "🤔", "off_topic": "↩️"
        }
        print(f"   [{emoji_map.get(cls, '?')} {cls}]", end="")
        if assessment.get("specific_error"):
            print(f" — {assessment['specific_error'][:60]}", end="")
        print()

        # ── Record in history ──────────────────────────────────────────────
        state.history.append({
            "turn": state.turn_count,
            "tutor": tutor_msg,
            "student": student_answer,
            "classification": cls,
            "specific_error": assessment.get("specific_error"),
            "concept_id": assessment.get("concept_assessed", state.current_focus_concept),
            "confidence": assessment.get("confidence")
        })

        # ── Update state ───────────────────────────────────────────────────
        update_state(state, assessment)

        # ── Pick next question if needed ───────────────────────────────────
        if cls == "correct" or (cls == "partially_correct" and state.concept_states[state.current_focus_concept].mastered):
            question = pick_next_question(state)
            state.current_question = question
        else:
            # Stay on current question for hint/decompose
            question = state.current_question or pick_next_question(state)

        if state.session_complete:
            tutor_resp = {"action": "end_session",
                          "message_to_student": "Excellent work! You've mastered all 8 BST concepts. Let me generate your session summary."}
            continue

        # ── Get tutor's next response ──────────────────────────────────────
        tutor_resp = get_tutor_response(state, question, assessment, student_answer)
        state.current_question = question

    # ── Session Summary ────────────────────────────────────────────────────
    print("\nGenerating session summary...")
    summary = generate_session_summary(state)
    print("\n" + "═" * 60)
    print(" SESSION SUMMARY")
    print("═" * 60)
    print(summary)
    print("═" * 60)

    # ── Mastery report ─────────────────────────────────────────────────────
    print("\n CONCEPT MASTERY REPORT")
    print("-" * 60)
    for concept in CONCEPT_MAP:
        cid = concept["id"]
        cs = state.concept_states[cid]
        icon = "✅" if cs.mastered else ("⚠️ " if cs.wrong_count >= 2 else "⬜")
        print(f"  {icon} {concept['name']}")
        print(f"     correct={cs.correct_count:.1f}  wrong={cs.wrong_count:.1f}  mastered={cs.mastered}")

    # ── Save transcript ────────────────────────────────────────────────────
    if save_transcript:
        import datetime
        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        fname = f"transcript_{ts}.json"
        transcript_data = {
            "metadata": {
                "topic": "Binary Search Trees",
                "timestamp": ts,
                "total_turns": state.turn_count,
                "concepts_mastered": state.mastered_count(),
                "total_concepts": len(CONCEPT_MAP)
            },
            "concept_mastery": {
                cid: {
                    "name": CONCEPT_BY_ID[cid]["name"],
                    "mastered": cs.mastered,
                    "correct_count": cs.correct_count,
                    "wrong_count": cs.wrong_count
                }
                for cid, cs in state.concept_states.items()
            },
            "history": state.history,
            "summary": summary
        }
        with open(fname, "w") as f:
            json.dump(transcript_data, f, indent=2)
        print(f"\n💾 Transcript saved: {fname}")

    return state


print("✅ Full session loop defined")
print("Run Cell 10 to start a tutoring session.")

## Cell 10 — ▶️ START TUTORING SESSION

Run this cell to begin an interactive session. Type your answers in the input box that appears.

> **Tip:** Type `quit` to end the session early and still receive your summary.

In [ ]:
# ── RUN THE TUTOR ─────────────────────────────────────────────────────────────
session_state = run_tutoring_session(save_transcript=True)

## Cell 11 — Stage 8: Analyze saved transcripts

In [ ]:
import glob

transcripts = sorted(glob.glob("transcript_*.json"))
print(f"Found {len(transcripts)} saved transcript(s):\n")

for fname in transcripts:
    with open(fname) as f:
        data = json.load(f)
    meta = data["metadata"]
    print(f"📄 {fname}")
    print(f"   Turns: {meta['total_turns']} | Concepts mastered: {meta['concepts_mastered']}/{meta['total_concepts']}")

    # Classification breakdown
    history = data.get("history", [])
    from collections import Counter
    counts = Counter(h.get("classification") for h in history)
    print(f"   Assessments: ", " | ".join(f"{k}: {v}" for k, v in counts.items()))
    print()

## Cell 12 — Unit test: Single-turn cycle

Test the full pipeline (Tutor → Assessor → State Update) without interactive input.

In [ ]:
print("Running single-turn integration test...")
print("-" * 50)

# Create a fresh state
test_state = SessionState()
test_q = pick_next_question(test_state)

print(f"1. First question selected: [{test_q['difficulty']}] {test_q['text']}")

# Simulate a correct answer
test_answer = "A tree is made of nodes connected by edges, with a root at the top."
print(f"2. Simulated student answer: '{test_answer}'")

# Assess
assessment = assess_response(test_q, test_answer, use_dev_model=True)
print(f"3. Assessor classification: {assessment['classification']} (confidence: {assessment['confidence']})")

# Update state
update_state(test_state, assessment)
print(f"4. State updated — {test_state.current_focus_concept} correct_count: {test_state.concept_states[test_q['concept_id']].correct_count}")

# Get tutor response
tutor_resp = get_tutor_response(test_state, test_q, assessment, test_answer)
print(f"5. Tutor action: {tutor_resp.get('action')}")
print(f"6. Tutor message: {tutor_resp.get('message_to_student', '')[:120]}...")

print("\n✅ Single-turn integration test passed")

## Cell 13 — Prompt versioning log

Project guide requirement: save tutor prompt versions with notes.

In [ ]:
PROMPT_VERSIONS = [
    {
        "version": "v1.0",
        "date": "initial",
        "tutor_model": TUTOR_MODEL,
        "assessor_model": ASSESSOR_MODEL,
        "change": "Initial prompt — basic Socratic instructions",
        "observation": "(fill after testing) Did the tutor withhold answers? Was hint quality good?"
    }
    # Add entries as you iterate:
    # {"version": "v1.1", "date": "...", "change": "Added 3 few-shot examples of 'just tell me' redirection", "observation": "..."}
]

print("Prompt version log:")
for v in PROMPT_VERSIONS:
    print(f"\n  [{v['version']}] {v['change']}")
    print(f"   Models: Tutor={v['tutor_model']}, Assessor={v['assessor_model']}")
    print(f"   Observation: {v['observation']}")

# Save versions
with open("prompt_versions.json", "w") as f:
    json.dump({
        "versions": PROMPT_VERSIONS,
        "tutor_system_prompt_current": TUTOR_SYSTEM_PROMPT,
        "assessor_system_prompt_current": ASSESSOR_SYSTEM_PROMPT
    }, f, indent=2)

print("\n✅ Prompt versions saved to prompt_versions.json")

## Cell 14 — Known failure modes (document as you find them)

Project guide strong requirement: honest documentation of where the system fails.

In [ ]:
DOCUMENTED_FAILURE_MODES = [
    {
        "mode": "Tutor gives answer under persistent pressure",
        "description": "After 3+ turns of student saying 'just tell me', Tutor occasionally caves and gives the answer despite explicit instructions.",
        "frequency": "~8% of sessions (to be measured)",
        "mitigation": "Added 'Under NO CIRCUMSTANCES' language. Consider few-shot examples in prompt.",
        "status": "Known, documented, partially mitigated"
    },
    {
        "mode": "Hint too strong",
        "description": "Hints sometimes contain the answer phrased as a leading question. E.g., 'Does left<root<right sound familiar?'",
        "frequency": "Occasionally on complexity questions",
        "mitigation": "Added hint strength test to prompt. Monitor transcripts.",
        "status": "Known, partially mitigated"
    },
    {
        "mode": "Assessor too lenient on partially_correct",
        "description": "Assessor tends to classify vague answers as partially_correct rather than incorrect.",
        "frequency": "~15% of incorrect answers",
        "mitigation": "Added 'default to incorrect' rule in Assessor prompt.",
        "status": "Known, mitigated — verify with calibration"
    },
    {
        "mode": "Concept stuck loop",
        "description": "Without MAX_TURNS_PER_CONCEPT, Tutor can get stuck asking variations forever if student never masters a concept.",
        "frequency": "Would occur without mitigation",
        "mitigation": "MAX_TURNS_PER_CONCEPT=5 hardcoded. System moves on and notes 'incomplete mastery'.",
        "status": "Resolved by design"
    }
]

print("DOCUMENTED FAILURE MODES")
print("=" * 60)
for i, fm in enumerate(DOCUMENTED_FAILURE_MODES, 1):
    print(f"\n{i}. {fm['mode']}")
    print(f"   {fm['description']}")
    print(f"   Frequency : {fm['frequency']}")
    print(f"   Mitigation: {fm['mitigation']}")
    print(f"   Status    : {fm['status']}")

with open("failure_modes.json", "w") as f:
    json.dump(DOCUMENTED_FAILURE_MODES, f, indent=2)

print("\n✅ Failure modes saved to failure_modes.json")

---
## Quick Reference

| Cell | Stage | What it does |
|------|-------|--------------|
| 1–2  | Setup | Install deps, set API key |
| 3    | Stage 0 | Verify NVIDIA NIM API |
| 4    | Stage 1 | Concept map + question pool (BST, 8 concepts, 24 questions) |
| 5    | Stage 2 | Assessor LLM + calibration (10 hand-crafted pairs) |
| 6    | Stage 3–4 | Session state, state updater, question selector |
| 7    | Stage 3 | Tutor LLM (Socratic, witholds answers) |
| 8    | Stage 7 | Session summary generator |
| 9    | Stage 5–6 | Full interactive loop with hint logic |
| **10** | **▶️** | **Run the tutoring session** |
| 11   | Stage 8 | Analyze saved transcripts |
| 12   | Test | Integration test (single turn, no input needed) |
| 13   | Admin | Prompt versioning log |
| 14   | Report | Documented failure modes |

### To change the topic
1. Edit Cell 4 — replace `CONCEPT_MAP` and `QUESTION_POOL` with your topic
2. Edit the `TUTOR_SYSTEM_PROMPT` in Cell 7 — update the topic name
3. Re-run all cells from Cell 4 onward

### Token economy (per project guide)
- Dev/test model (`DEV_MODEL`): use for calibration and prompt iteration
- Production models: use for final student sessions
- Estimated: ~16 credits/turn × 20 turns = ~320 credits/session